In [ ]:
from investing import data, history
from importlib import reload

reload(data)

In [ ]:
data.load_dividends("../data/5-way-dividends.xlsx")

In [ ]:
data.load_prices("../data/5-way-prices.xlsx")

In [ ]:
history.load_market_history("../data/5-way-prices.xlsx", "../data/5-way-dividends.xlsx")

In [ ]:
from datetime import date
import polars as pl

from investing import analysis as a
from investing.history import load_market_history
from investing.portfolio import AssetAllocation, HoldingTarget
from investing.simulation import AnnualRebalance, monthly_time_step, simulate

reload(a)


history = load_market_history(
    "../data/5-way-prices.xlsx", "../data/5-way-dividends.xlsx"
)
strategy = AnnualRebalance(
    AssetAllocation([HoldingTarget(ticker, 1) for ticker in history.securities.keys()]),
    0.05,
)
results = simulate(history, date(2022, 1, 1), 100_000, strategy, monthly_time_step)

positions = a.value_history(results).with_columns(
    (pl.col("price") * pl.col("quantity")).round(2).alias("valuation")
)
positions

In [ ]:
values = pl.concat(
    [
        positions.select("date", "ticker", "valuation"),
        positions.group_by("date")
        .agg(pl.sum("valuation"))
        .with_columns(pl.lit("TOTAL").alias("ticker"))
        .select("date", "ticker", "valuation"),
    ]
)
values

In [ ]:
import seaborn as sns

sns.lineplot(values, x="date", y="valuation", hue="ticker")